# SQL — Agregação de dados

Neste material vamos trabalhar com **agregação de dados** utilizando PostgreSQL.

O foco será transformar vários registros em informações resumidas, utilizando funções de agregação e agrupamentos.

Serão apresentados:

- `COUNT`;
- `SUM`;
- `AVG`;
- `MIN` e `MAX`;
- `GROUP BY`;
- `HAVING`;
- combinação entre `WHERE`, `GROUP BY` e `HAVING`;
- `DISTINCT` com funções de agregação;
- `ORDER BY` em resultados agregados;
- cuidados com `NULL`;
- exercícios práticos.

Vamos continuar utilizando as tabelas do nosso modelo de exemplo:

```text
clientes
produtos
pedidos
itens_pedido
```

A ideia é utilizar o mesmo banco trabalhado nos materiais anteriores.


# O que é agregação?

Uma consulta comum retorna registros individuais.

Por exemplo:

```sql
SELECT *
FROM produtos;
```

Nesse caso, cada linha representa um produto.

Em uma consulta de **agregação**, queremos obter informações resumidas sobre vários registros.

Por exemplo:

- quantos produtos existem;
- qual é o maior preço;
- qual é o menor preço;
- qual é o preço médio;
- qual é o valor total de determinados registros.

Para isso, utilizamos **funções de agregação**.

As principais funções são:

```text
COUNT  → quantidade
SUM    → soma
AVG    → média
MIN    → menor valor
MAX    → maior valor
```


# COUNT

A função `COUNT` permite contar registros.

A forma mais simples é:

```sql
SELECT COUNT(*)
FROM produtos;
```

O resultado será a quantidade de registros existentes na tabela `produtos`.

Também podemos utilizar um alias para deixar o resultado mais claro:

```sql
SELECT COUNT(*) AS quantidade_produtos
FROM produtos;
```

O `AS` define o nome da coluna retornada pela consulta.


## COUNT e uma coluna

Também podemos contar os valores de uma coluna:

```sql
SELECT COUNT(nome)
FROM produtos;
```

Nesse caso, `COUNT(nome)` conta apenas os registros em que `nome` não é `NULL`.

Essa diferença é importante:

```text
COUNT(*)       → conta as linhas
COUNT(coluna)  → conta valores não NULL da coluna
```


# SUM

A função `SUM` realiza a soma dos valores de uma coluna numérica.

Por exemplo, para somar o estoque dos produtos:

```sql
SELECT SUM(estoque) AS estoque_total
FROM produtos;
```

O resultado representa a quantidade total armazenada considerando todos os produtos.


Podemos utilizar `SUM` junto com `WHERE`.

Por exemplo, para somar apenas o estoque dos produtos com preço maior que `100`:

```sql
SELECT SUM(estoque) AS estoque_total
FROM produtos
WHERE preco > 100;
```

O `WHERE` filtra as linhas antes da agregação.


# AVG

A função `AVG` calcula a média dos valores.

Por exemplo:

```sql
SELECT AVG(preco) AS preco_medio
FROM produtos;
```

O resultado representa o preço médio dos produtos.

Podemos combinar com `WHERE`:

```sql
SELECT AVG(preco) AS preco_medio
FROM produtos
WHERE estoque > 0;
```


# MIN e MAX

As funções `MIN` e `MAX` permitem encontrar, respectivamente, o menor e o maior valor.

Por exemplo:

```sql
SELECT
    MIN(preco) AS menor_preco,
    MAX(preco) AS maior_preco
FROM produtos;
```

Podemos utilizar várias funções de agregação na mesma consulta.


# Utilizando várias funções de agregação

Podemos combinar diferentes funções em uma única consulta:

```sql
SELECT
    COUNT(*) AS quantidade_produtos,
    SUM(estoque) AS estoque_total,
    AVG(preco) AS preco_medio,
    MIN(preco) AS menor_preco,
    MAX(preco) AS maior_preco
FROM produtos;
```

Essa consulta transforma vários registros em um único resumo.


# GROUP BY

As funções de agregação podem produzir um resumo geral da tabela.

Mas muitas vezes queremos criar um resumo **por categoria ou grupo**.

É nesse momento que utilizamos o `GROUP BY`.

A estrutura básica é:

```sql
SELECT coluna,
       função_de_agregação(...)
FROM tabela
GROUP BY coluna;
```

Por exemplo, considerando uma tabela `pedidos` que possui uma coluna `cliente_id`:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
GROUP BY cliente_id;
```

Nesse caso, teremos uma linha para cada `cliente_id`, mostrando a quantidade de pedidos daquele cliente.


## GROUP BY com mais de uma coluna

Também podemos agrupar utilizando mais de uma coluna.

Por exemplo:

```sql
SELECT
    cliente_id,
    status,
    COUNT(*) AS quantidade
FROM pedidos
GROUP BY
    cliente_id,
    status;
```

Nesse caso, cada combinação de `cliente_id` e `status` forma um grupo.


# Regra importante do GROUP BY

Quando utilizamos `GROUP BY`, as colunas selecionadas precisam fazer sentido em relação ao agrupamento.

Por exemplo:

```sql
SELECT
    cliente_id,
    COUNT(*)
FROM pedidos
GROUP BY cliente_id;
```

`cliente_id` está no `GROUP BY` e `COUNT(*)` é uma função de agregação.

Já uma consulta como:

```sql
SELECT
    cliente_id,
    status,
    COUNT(*)
FROM pedidos
GROUP BY cliente_id;
```

não é válida no PostgreSQL se `status` não estiver agrupado nem for utilizado dentro de uma função de agregação.

Uma forma válida seria:

```sql
SELECT
    cliente_id,
    status,
    COUNT(*)
FROM pedidos
GROUP BY
    cliente_id,
    status;
```


# WHERE antes da agregação

O `WHERE` é utilizado para filtrar os registros **antes** do agrupamento.

Por exemplo:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
WHERE status = 'FINALIZADO'
GROUP BY cliente_id;
```

A lógica pode ser entendida assim:

```text
1. FROM
2. WHERE
3. GROUP BY
4. agregação
5. SELECT
```

Portanto, o `WHERE` trabalha sobre as linhas que serão agrupadas.


# HAVING

O `HAVING` permite filtrar os **grupos** produzidos pelo `GROUP BY`.

Por exemplo, queremos mostrar somente os clientes que possuem mais de 2 pedidos:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
GROUP BY cliente_id
HAVING COUNT(*) > 2;
```

Aqui temos uma diferença importante:

```text
WHERE  → filtra registros antes do agrupamento
HAVING → filtra grupos depois do agrupamento
```


## WHERE x HAVING

Considere:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
WHERE status = 'FINALIZADO'
GROUP BY cliente_id
HAVING COUNT(*) >= 2;
```

A consulta faz duas coisas:

1. considera apenas pedidos com `status = 'FINALIZADO'`;
2. depois agrupa os pedidos por cliente;
3. por fim, mantém apenas os grupos com pelo menos 2 pedidos.

Essa diferença é fundamental para trabalhar com agregações.


# COUNT(DISTINCT ...)

Podemos combinar `COUNT` com `DISTINCT`.

Por exemplo, para contar quantos clientes diferentes aparecem nos pedidos:

```sql
SELECT COUNT(DISTINCT cliente_id) AS clientes_com_pedidos
FROM pedidos;
```

O `DISTINCT` elimina valores repetidos antes da contagem.

Isso é diferente de:

```sql
SELECT COUNT(cliente_id)
FROM pedidos;
```

O segundo comando conta os valores de `cliente_id` em cada registro, mesmo quando o mesmo cliente aparece várias vezes.


# Agregação com ORDER BY

Também podemos ordenar resultados agregados.

Por exemplo:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
GROUP BY cliente_id
ORDER BY quantidade_pedidos DESC;
```

Nesse exemplo, os clientes com maior quantidade de pedidos aparecem primeiro.

Também podemos ordenar diretamente pela expressão:

```sql
ORDER BY COUNT(*) DESC;
```


# Agregação e NULL

As funções de agregação possuem comportamentos importantes em relação a `NULL`.

Por exemplo:

```sql
AVG(preco)
```

considera os valores não `NULL` da coluna.

Da mesma forma:

```sql
SUM(preco)
```

ignora valores `NULL`.

Já:

```sql
COUNT(preco)
```

conta apenas os valores não `NULL`.

Enquanto:

```sql
COUNT(*)
```

conta as linhas.


# Exemplo completo

Podemos combinar vários recursos em uma mesma consulta.

Considere a tabela `pedidos`:

```sql
SELECT
    cliente_id,
    COUNT(*) AS quantidade_pedidos
FROM pedidos
WHERE status = 'FINALIZADO'
GROUP BY cliente_id
HAVING COUNT(*) >= 2
ORDER BY quantidade_pedidos DESC;
```

A consulta pode ser interpretada da seguinte maneira:

```text
FROM
    → utiliza a tabela pedidos

WHERE
    → mantém somente pedidos finalizados

GROUP BY
    → separa os pedidos por cliente

COUNT
    → conta os pedidos de cada cliente

HAVING
    → mantém clientes com pelo menos 2 pedidos

ORDER BY
    → ordena do maior para o menor número de pedidos
```


# Agregando valores dos itens de pedido

A tabela `itens_pedido` pode ser utilizada para calcular informações relacionadas aos itens vendidos.

Supondo que ela possua:

```text
pedido_id
produto_id
quantidade
preco_unitario
```

podemos calcular o valor de cada item:

```sql
SELECT
    pedido_id,
    produto_id,
    quantidade,
    preco_unitario,
    quantidade * preco_unitario AS subtotal
FROM itens_pedido;
```

O `subtotal` é uma expressão calculada para cada registro.


## Somando os valores

Podemos utilizar `SUM` sobre uma expressão.

Por exemplo:

```sql
SELECT
    pedido_id,
    SUM(quantidade * preco_unitario) AS valor_total
FROM itens_pedido
GROUP BY pedido_id;
```

Agora temos o valor total de cada pedido.

Essa é uma aplicação muito comum de agregação: transformar vários itens em um resumo por pedido.


# Cuidados com agregação

Alguns cuidados são importantes:

### 1. Não confundir WHERE e HAVING

```text
WHERE  → linhas
HAVING → grupos
```

### 2. Verificar o que está sendo contado

```sql
COUNT(*)
COUNT(coluna)
COUNT(DISTINCT coluna)
```

produzem resultados diferentes.

### 3. Observar NULL

Funções como `SUM`, `AVG` e `COUNT(coluna)` não tratam `NULL` da mesma forma que `COUNT(*)`.

### 4. Pensar no nível do agrupamento

Antes de escrever o `GROUP BY`, pergunte:

> "Uma linha do resultado representa o quê?"

Por exemplo:

```text
uma linha por cliente
uma linha por pedido
uma linha por produto
uma linha por cliente e status
```

Essa pergunta ajuda a definir corretamente o agrupamento.


# Resumo

Neste material aprendemos a resumir dados utilizando funções de agregação.

Principais funções:

```text
COUNT → quantidade
SUM   → soma
AVG   → média
MIN   → menor valor
MAX   → maior valor
```

Para criar grupos:

```sql
GROUP BY
```

Para filtrar os registros antes do agrupamento:

```sql
WHERE
```

Para filtrar os grupos depois da agregação:

```sql
HAVING
```

Para contar valores distintos:

```sql
COUNT(DISTINCT coluna)
```

Uma das estruturas mais importantes deste conteúdo é:

```sql
SELECT
    coluna,
    COUNT(*) AS quantidade
FROM tabela
WHERE condição
GROUP BY coluna
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;
```

A agregação é uma das bases para transformar dados armazenados em informações úteis para análise.
